- This notebook is a wip of the model engineering task, fine-tuning transformer model such as BERT and RoBERTa.
- Next Steps:
    - Model optimisation for inference (reducing latency)
    - Per-entity evaluation
    - Error analysis
    - Best model export

Code source -> [article](https://medium.com/@whyamit101/fine-tuning-bert-for-named-entity-recognition-ner-b42bcf55b51d)

### **PIPELINE CONSTRUCTION**

STEP 1: Group by `resume_idx`

STEP 2: Merge overlapping chunks

STEP 3: Reconstruct full sequence

STEP 4: Compute metrics

### IMPORTANT!!!!
**Notes for training:**
- Each resume produces 1-N chunks. Use `resume_idx` to re-group predictions per resume at inference.
- Overlapping tokens (stride region) are masked with -100 — each token contributes to the loss exactly once.
- O-tag rate is ~86-92% — class imbalance is expected in NER.

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')
import sys
sys.path.append("..")  # Add project root so 'src' is importable


In [ ]:
# Uncomment in Colab / fresh environment
#!pip -q install datasets transformers seqeval scikit-learn pandas accelerate>=1.1.0

### **IMPORT DEPENDENCIES**

In [1]:
import os
import csv
import time
from datetime import datetime

import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from collections import defaultdict, Counter

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)

from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

ModuleNotFoundError: No module named 'torch'

In [ ]:
import warnings
warnings.filterwarnings("ignore")

### **1. CONFIG**

In [ ]:
# ==================== MODEL ====================
MODEL_NAME = "bert-base-uncased"
# MODEL_NAME = "roberta-base"           # Uncomment this line for RoBERTa fine-tuning

# ==================== PATH ====================

DATASET_PATH = "../data/processed/resume_ner_hf"

OUT_DIR = f"../checkpoints/{MODEL_NAME}"
LOG_DIR = f"../logs/{MODEL_NAME}"
RESULT_CSV = "../results/experiment_results.csv"
ENTITY_CSV = "../results/per_entity_results.csv"

"""
DATASET_PATH = "/content/drive/MyDrive/resume_ner_hf"

OUT_DIR = f"/content/checkpoints/{MODEL_NAME}"      # Uncomment for colab
LOG_DIR = f"/content/logs/{MODEL_NAME}"
RESULT_CSV = "/content/results/experiment_results.csv"
ENTITY_CSV = "/content/results/per_entity_results.csv"
"""

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
#os.makedirs("/content/results", exist_ok=True)         # Uncomment for colab
os.makedirs("../results", exist_ok=True)

# ==================== MODEL HYPERPARAMETERS ====================
LEARNING_RATE = 2e-5        # Best LR: 2e-5 ~ 3e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
EPOCH = 4           # Best epoch: 4, 5 (but model overfitting early stopping at 2, 3)
WEIGHT_DECAY = 0.01         # havnt played around 
WARMUP_RATIO = 0.1
STRIDE = 128        # Must match dataset

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

### **2. LOAD DATASET**

In [ ]:
ds = load_from_disk(DATASET_PATH)

# ==================== Label Setup ====================
# label should be manually define BIO tag
label_list = [
    "O",
    "B-JOB_TITLE",
    "I-JOB_TITLE",
    "B-SKILL",
    "I-SKILL",
    "B-EDUCATION",
    "I-EDUCATION",
]

id2label = {i: str(label) for i, label in enumerate(label_list)}
label2id = {str(label): i for i, label in enumerate(label_list)}

print("Data loaded successfully!")
print("Labels:", id2label)

### **3. MODELLING**

In [ ]:
# ==================== Load Model ====================
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)           # for padding purposes

# ==================== Data Collator ====================
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)      # dataset is tokenized, still needed for padding

# ==================== Metrics ====================
# chunck-level for best checkpoint selection (training)
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    true_labels = []
    true_preds = []

    for label_seq, pred_seq in zip(labels, preds):

        seq_labels = []
        seq_preds = []

        for l, p in zip(label_seq, pred_seq):

            if l == -100:           # ignore -100
                continue

            seq_labels.append(id2label[l])
            seq_preds.append(id2label[p])

        true_labels.append(seq_labels)
        true_preds.append(seq_preds)

    precision = precision_score(true_labels, true_preds)
    recall = recall_score(true_labels, true_preds)
    f1 = f1_score(true_labels, true_preds)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Sanity check
print(id2label)
print(ds["train"][0]["labels"])     # dataset labels should be integers

### **4. TRAINING PIPELINE**

In [ ]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCH,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_dir=LOG_DIR,
    logging_strategy="epoch",

    fp16=torch.cuda.is_available(),
    seed=42
)

# ==================== Trainer ====================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,                                     
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)


### **Post-Processing**


*   Fix broken BIO sequence
*   O -> entity noise



In [ ]:
# ==================== BIO Correction Rules ====================
def fix_bio_sequence(seq):
    fixed = []
    prev_tag = "O"

    for tag in seq:
        if tag.startswith("I-"):
            entity = tag[2:]

            # If previous tag is not same entity → convert to B-
            if prev_tag == "O" or prev_tag[2:] != entity:
                tag = "B-" + entity

        fixed.append(tag)
        prev_tag = tag

    return fixed

# ==================== Noise Filter (Isolated entities [O B-SKILL O -> O O O]) ====================
def remove_isolated_entities(seq):
    fixed = seq.copy()

    for i in range(1, len(seq) - 1):
        if seq[i] != "O":
            if seq[i-1] == "O" and seq[i+1] == "O":
                fixed[i] = "O"

    return fixed
# ==================== Merged ====================
def fix_fragmented_entities(seq):
    fixed = seq.copy()

    for i in range(1, len(seq) - 1):
        if seq[i] == "O":
            prev_tag = seq[i-1]
            next_tag = seq[i+1]

            if prev_tag.startswith("B-") and next_tag.startswith("I-"):
                if prev_tag[2:] == next_tag[2:]:
                    fixed[i] = "I-" + prev_tag[2:]

    return fixed

# ==================== Post-Process ====================
def post_process_sequence(seq):
    seq = fix_bio_sequence(seq)
    seq = fix_fragmented_entities(seq)
    #seq = remove_isolated_entities(seq)      # Probably too aggressive
    return seq

In [ ]:
# ==================== Reseume-Level Evaluation Pipeline ====================
def merge_chunks(chunks, stride=128):
    merged = []

    for i, chunk in enumerate(chunks):
        if i == 0:
            merged.extend(chunk)
        else:
            merged.extend(chunk[stride:])       # This remove overlap

    return merged

def evaluate_resume_level(trainer, dataset, id2label, stride=128):
    preds_output = trainer.predict(dataset)

    logits = preds_output.predictions
    labels = preds_output.label_ids
    preds = np.argmax(logits, axis=2)

    resume_ids = dataset["resume_idx"]

    # Group by resume_idx
    grouped_preds = defaultdict(list)
    grouped_labels = defaultdict(list)

    for i, r_id in enumerate(resume_ids):
        grouped_preds[r_id].append(preds[i])
        grouped_labels[r_id].append(labels[i])

    # Merge chunks
    final_preds = []
    final_labels = []

    for r_id in grouped_preds:
        merged_pred = merge_chunks(grouped_preds[r_id], stride=stride)
        merged_label = merge_chunks(grouped_labels[r_id], stride=stride)

        seq_pred = []
        seq_label = []

        for p, l in zip(merged_pred, merged_label):
            if l == -100:
                continue
            seq_pred.append(id2label[int(p)])
            seq_label.append(id2label[int(l)])

        seq_pred = post_process_sequence(seq_pred)

        final_preds.append(seq_pred)
        final_labels.append(seq_label)

    # Metrics
    precision = precision_score(final_labels, final_preds)
    recall = recall_score(final_labels, final_preds)
    f1 = f1_score(final_labels, final_preds)

    report = classification_report(
        final_labels,
        final_preds,
        output_dict=True,
        zero_division=0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "report": report,
        "final_labels": final_labels,
        "final_preds": final_preds
    }

# ==================== Report Log ====================
def save_entity_report(report, model_name, split, file_path):
    rows = []

    for entity, scores in report.items():
        if isinstance(scores, dict) and "f1-score" in scores:
            rows.append({
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "model": model_name,
                "split": split,
                "entity": entity,
                "precision": scores["precision"],
                "recall": scores["recall"],
                "f1": scores["f1-score"],
                "support": scores["support"]
            })

    df = pd.DataFrame(rows)

    if os.path.exists(file_path):
        df.to_csv(file_path, mode="a", header=False, index=False)
    else:
        df.to_csv(file_path, index=False)

    return df

In [ ]:
# ==================== Train & Time Tracking ====================
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time
print(f"\nTraining Time: {training_time/60:.2f} minutes")

Model started to overfit a bit

In [ ]:
# ==================== Resume-Level Validation Eval ====================
val_results = evaluate_resume_level(
    trainer=trainer,
    dataset=ds["validation"],
    id2label=id2label,
    stride=STRIDE
)

print("\n===== Resume-Level Validation Results =====")
print(f"Precision: {val_results['precision']:.4f}")
print(f"Recall:    {val_results['recall']:.4f}")
print(f"F1:        {val_results['f1']:.4f}")

val_entity_df = save_entity_report(
    report=val_results["report"],
    model_name=MODEL_NAME,
    split="validation",
    file_path=ENTITY_CSV
)

display(val_entity_df)

**SUPPORT**

Example:

    B-SKILL I-SKILL I-SKILL -> 1 SKILL entity (NOT 3!)

Explanation:

- Support = Number of true occurrences of that entity in ground truth

- EDUCATION: Support = 201 -> Little data, harder to learn -> Explains low F1

- SKILL: Support High, F1 Low -> Model struggle with common class

- Comfirms class imbalance + semantic difficulty



precision > recall -> Model is too conservative

**Main bottleneck:**
- hard entity (EDUCATION) + imbalance
- O = ~86-92%, model predicts 'O' to be safe -> Low F1 

**Best Next Move:**
- Fine tune LR -> Higher LR, more aggressive learning
- More epochs -> Rare classes learned better
- Entity (EDUCATION) analysis with ground truth 


**Ground Truth Analysis**

In [ ]:
final_labels = val_results["final_labels"]
final_preds = val_results["final_preds"]

In [ ]:
# Sanity check
print(len(final_labels), len(final_preds))

In [ ]:
def collect_errors(true_labels, pred_labels):
    errors = []

    for t_seq, p_seq in zip(true_labels, pred_labels):
        for t, p in zip(t_seq, p_seq):
            if t != p:
                errors.append((t, p))

    return errors

errors = collect_errors(final_labels, final_preds)

In [ ]:
from collections import Counter

error_counts = Counter(errors)

print(error_counts.most_common(20))

In [ ]:
def show_examples(true_labels, pred_labels, n=5):
    count = 0

    for t_seq, p_seq in zip(true_labels, pred_labels):
        if t_seq != p_seq:
            print("\n--- Example ---")
            print("TRUE :", t_seq)
            print("PRED :", p_seq)

            count += 1
            if count >= n:
                break

show_examples(final_labels, final_preds)

#### Ground-Truth Analysis Main Takeaway (6th)

**Most common errors:**

- ('B-SKILL', 'O') -> 3855 : Missed skills
- ('O', 'B-SKILL') -> 3087 : False positive
- ('I-SKILL', 'O') -> 2689 : Broken spans
- ('O', 'I-SKILL') -> 1776 : Invalid starts

*Based on common errors, safe to say model is both too aggressive AND too conservatives*

**Low recall (conservative behavior)**

- B-SKILL -> O
- I-SKILL -> O

**Low precision (aggressive behavior)**

- O -> B-SKILL
- O -> I-SKILL

**CONCLUSION:**

- Model is unstable. The competing bahavior indicates instability in boundary detection for SKILL entities. WHY?
- SKILL entities account for most of the misclassifications (O <-> SKILL). Reflects semantic variability & context dependency of skill expressions. 
- SKILL very diverse (Python, teamwork, leadership, SQL, etc..) -> weak lexical patterns but high in frequency -> model cannot learn clean boundary (Semantic ambiguity)
- SINGLE TOKEN!!!
- HOW THE HECK DO WE SOLVE THIS!!? 
- OR do we just accept SKILL is noisy. we managed to get F1 ~0.6 (consider good for this dataset)

**Best Next Step (IF WE HV TIME)**
1. CRF Layer -> fix BIO issues
2. External SKILL lexicon (rule-based)

but roberta can improve our F1

### **5. TESTING**

Best practice -> use best model from checkpoint during test eval

In [ ]:
test_results = evaluate_resume_level(
    trainer=trainer,
    dataset=ds["test"],
    id2label=id2label,
    stride=STRIDE
)

print("\n===== Resume-Level Test Results =====")
print(f"Precision: {test_results['precision']:.4f}")
print(f"Recall:    {test_results['recall']:.4f}")
print(f"F1:        {test_results['f1']:.4f}")

test_entity_df = save_entity_report(
    report=test_results["report"],
    model_name=MODEL_NAME,
    split="test",
    file_path=ENTITY_CSV
)

display(test_entity_df)

In [ ]:
def append_row_to_csv(file_path, row):
    file_exists = os.path.exists(file_path)

    with open(file_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)

if device == "cuda":
    max_gpu_mem = torch.cuda.max_memory_allocated(0) / 1e9
else:
    max_gpu_mem = 0

log_data = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model":MODEL_NAME,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "validation_precision": val_results["precision"],
    "validation_recall": val_results["recall"],
    "validation_f1": val_results["f1"],
    "test_precision": test_results["precision"],
    "test_recall": test_results["recall"],
    "test_f1": test_results["f1"],
    "training_time_sec": round(training_time, 2),
    "training_time_min": round(training_time / 60, 2),
    "max_gpu_memory_gb": round(max_gpu_mem, 2),
    "device": device
}

append_row_to_csv(RESULT_CSV, log_data)

print(f"\nExperiment logged to: {RESULT_CSV}")
print(f"Per-entity results logged to: {ENTITY_CSV}")

**For Colab**

In [ ]:
# if using colab
#!cp -r /content/results /content/drive/MyDrive/cv_project_results

In [ ]:
#!rm -r /content/results/
#!rm -r /content/logs/
#!rm -r /content/checkpoints     # delete old checkpoint for retraining to avoid mismatch